In [ ]:
# ============================================================
#  Federated Learning for Potato Plant Disease Classification
#  Algorithms: FedAvg | FedProx
#  Dataset   : Potato Plant Diseases Data (Kaggle)
#  Author    : Generated from paper specifications
# ============================================================

# ── 0. Install / imports ────────────────────────────────────
import os, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models

from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             classification_report)

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ── 1. Config ───────────────────────────────────────────────
class Config:
    # Paths  (Kaggle input directory)
    DATA_DIR        = "/kaggle/input/datasets/hafiznouman786/potato-plant-diseases-data/PotatoPlants"   # adjust if subfolder differs

    # Federated settings  (paper §4.1 – §4.4)
    NUM_CLIENTS     = 5
    DIRICHLET_ALPHA = 0.5        # non-IID heterogeneity
    NUM_ROUNDS      = 50         # communication rounds
    LOCAL_EPOCHS    = 5
    BATCH_SIZE      = 32
    LR              = 0.001
    MOMENTUM        = 0.9        # SGD momentum

    # FedProx
    MU              = 0.01       # proximal coefficient

    # Device
    DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    IMG_SIZE        = 224

cfg = Config()
print(f"Device: {cfg.DEVICE}")


# ── 2. Discover dataset root ────────────────────────────────
def find_image_root(base: str) -> str:
    """Walk subdirs until we find a folder containing sub-folders (classes)."""
    for root, dirs, files in os.walk(base):
        # A valid image folder has sub-directories each holding images
        subdirs = [d for d in dirs if not d.startswith('.')]
        if subdirs:
            # check at least one subdir has images
            sample = os.path.join(root, subdirs[0])
            imgs = [f for f in os.listdir(sample)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            if imgs:
                return root
    return base

DATA_ROOT = find_image_root(cfg.DATA_DIR)
print(f"Image root found: {DATA_ROOT}")


# ── 3. Transforms ───────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


# ── 4. Load full dataset & split train/test ──────────────────
full_dataset = datasets.ImageFolder(root=DATA_ROOT, transform=train_transform)
CLASS_NAMES  = full_dataset.classes
NUM_CLASSES  = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Total samples: {len(full_dataset)}")

# 80/20 train-test split (stratified by label)
from sklearn.model_selection import train_test_split

targets = np.array(full_dataset.targets)
all_idx = np.arange(len(targets))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42, stratify=targets)

# Test set uses test_transform
class SubsetWithTransform(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.transform:
            # img is already a tensor here from parent transform;
            # we need the PIL image. Re-load from path instead.
            path, _ = self.dataset.samples[self.indices[idx]]
            from PIL import Image
            img = Image.open(path).convert("RGB")
            img = self.transform(img)
        return img, label

test_dataset  = SubsetWithTransform(full_dataset, test_idx, transform=test_transform)
train_dataset = Subset(full_dataset, train_idx)

print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")


# ── 5. Dirichlet non-IID partition ─────────────────────────
def dirichlet_partition(dataset, num_clients, alpha, num_classes):
    """
    Partition dataset indices among clients using Dirichlet(alpha).
    Returns dict {client_id: [indices]}
    """
    targets = np.array([dataset.dataset.targets[i] for i in dataset.indices])
    class_indices = defaultdict(list)
    for idx, label in enumerate(targets):
        class_indices[label].append(idx)

    client_indices = defaultdict(list)
    for cls in range(num_classes):
        cls_idx = np.array(class_indices[cls])
        np.random.shuffle(cls_idx)
        # Sample proportions from Dirichlet
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        # Ensure no client gets 0 samples
        proportions = np.maximum(proportions, 1e-6)
        proportions /= proportions.sum()
        splits = (proportions * len(cls_idx)).astype(int)
        # Fix rounding
        splits[-1] = len(cls_idx) - splits[:-1].sum()
        splits = np.maximum(splits, 0)

        start = 0
        for cid, count in enumerate(splits):
            client_indices[cid].extend(cls_idx[start:start + count].tolist())
            start += count

    return dict(client_indices)

client_data_map = dirichlet_partition(train_dataset, cfg.NUM_CLIENTS,
                                      cfg.DIRICHLET_ALPHA, NUM_CLASSES)

print("\n── Client data distribution ──")
for cid, idxs in client_data_map.items():
    local_labels = [train_dataset.dataset.targets[train_dataset.indices[i]] for i in idxs]
    dist = {CLASS_NAMES[c]: local_labels.count(c) for c in range(NUM_CLASSES)}
    print(f"  Client {cid}: {len(idxs)} samples | {dist}")


# ── 6. DataLoaders per client ───────────────────────────────
def get_client_loader(client_id):
    indices  = client_data_map[client_id]
    subset   = Subset(train_dataset, indices)
    return DataLoader(subset, batch_size=cfg.BATCH_SIZE,
                      shuffle=True, num_workers=2, pin_memory=True)

client_loaders = {cid: get_client_loader(cid) for cid in range(cfg.NUM_CLIENTS)}
test_loader    = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)


# ── 7. Model ────────────────────────────────────────────────
def build_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(cfg.DEVICE)


# ── 8. Evaluation helper ─────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(cfg.DEVICE), labels.to(cfg.DEVICE)
            outputs = model(imgs)
            preds   = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    return acc, prec, rec, f1, all_labels, all_preds


# ── 9. FedAvg ───────────────────────────────────────────────
def fedavg_local_train(model, loader, epochs):
    model.train()
    optimizer  = optim.SGD(model.parameters(), lr=cfg.LR, momentum=cfg.MOMENTUM)
    criterion  = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(cfg.DEVICE), labels.to(cfg.DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
    return model.state_dict()


def fedavg_aggregate(global_sd, client_sds, client_sizes):
    total = sum(client_sizes)
    new_sd = copy.deepcopy(global_sd)
    for key in new_sd:
        new_sd[key] = torch.zeros_like(new_sd[key], dtype=torch.float32)
        for sd, size in zip(client_sds, client_sizes):
            new_sd[key] += sd[key].float() * (size / total)
    return new_sd


def run_fedavg():
    print("\n" + "="*60)
    print("  Running FedAvg")
    print("="*60)
    global_model = build_model(NUM_CLASSES)
    history = {"round": [], "accuracy": [], "precision": [],
                "recall": [], "f1": []}

    for rnd in range(1, cfg.NUM_ROUNDS + 1):
        client_sds, client_sizes = [], []

        for cid in range(cfg.NUM_CLIENTS):
            local_model = build_model(NUM_CLASSES)
            local_model.load_state_dict(copy.deepcopy(global_model.state_dict()))
            sd = fedavg_local_train(local_model, client_loaders[cid], cfg.LOCAL_EPOCHS)
            client_sds.append(sd)
            client_sizes.append(len(client_data_map[cid]))

        agg_sd = fedavg_aggregate(global_model.state_dict(), client_sds, client_sizes)
        global_model.load_state_dict(agg_sd)

        if rnd % 5 == 0 or rnd == 1:
            acc, prec, rec, f1, _, _ = evaluate(global_model, test_loader)
            history["round"].append(rnd)
            history["accuracy"].append(acc)
            history["precision"].append(prec)
            history["recall"].append(rec)
            history["f1"].append(f1)
            print(f"  Round {rnd:3d} | Acc={acc:.4f}  Prec={prec:.4f}  "
                  f"Rec={rec:.4f}  F1={f1:.4f}")

    return global_model, history


# ── 10. FedProx ─────────────────────────────────────────────
def fedprox_local_train(model, loader, global_sd, epochs, mu):
    model.train()
    optimizer  = optim.SGD(model.parameters(), lr=cfg.LR, momentum=cfg.MOMENTUM)
    criterion  = nn.CrossEntropyLoss()

    # Keep a frozen copy of global params on device
    global_params = {k: v.to(cfg.DEVICE).float() for k, v in global_sd.items()}

    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(cfg.DEVICE), labels.to(cfg.DEVICE)
            optimizer.zero_grad()
            outputs  = model(imgs)
            ce_loss  = criterion(outputs, labels)

            # Proximal term  ½·μ·||w − w_global||²
            prox_term = 0.0
            for name, param in model.named_parameters():
                if name in global_params:
                    prox_term += ((param - global_params[name]) ** 2).sum()
            loss = ce_loss + (mu / 2.0) * prox_term

            loss.backward()
            optimizer.step()
    return model.state_dict()


def run_fedprox():
    print("\n" + "="*60)
    print("  Running FedProx  (μ = {})".format(cfg.MU))
    print("="*60)
    global_model = build_model(NUM_CLASSES)
    history = {"round": [], "accuracy": [], "precision": [],
                "recall": [], "f1": []}

    for rnd in range(1, cfg.NUM_ROUNDS + 1):
        global_sd    = copy.deepcopy(global_model.state_dict())
        client_sds, client_sizes = [], []

        for cid in range(cfg.NUM_CLIENTS):
            local_model = build_model(NUM_CLASSES)
            local_model.load_state_dict(copy.deepcopy(global_sd))
            sd = fedprox_local_train(local_model, client_loaders[cid],
                                     global_sd, cfg.LOCAL_EPOCHS, cfg.MU)
            client_sds.append(sd)
            client_sizes.append(len(client_data_map[cid]))

        agg_sd = fedavg_aggregate(global_sd, client_sds, client_sizes)
        global_model.load_state_dict(agg_sd)

        if rnd % 5 == 0 or rnd == 1:
            acc, prec, rec, f1, _, _ = evaluate(global_model, test_loader)
            history["round"].append(rnd)
            history["accuracy"].append(acc)
            history["precision"].append(prec)
            history["recall"].append(rec)
            history["f1"].append(f1)
            print(f"  Round {rnd:3d} | Acc={acc:.4f}  Prec={prec:.4f}  "
                  f"Rec={rec:.4f}  F1={f1:.4f}")

    return global_model, history


# ── 11. Train both algorithms ────────────────────────────────
fedavg_model,  fedavg_hist  = run_fedavg()
fedprox_model, fedprox_hist = run_fedprox()


# ── 12. Final evaluation ─────────────────────────────────────
print("\n" + "="*60)
print("  FINAL TEST RESULTS")
print("="*60)

results = {}
for name, model in [("FedAvg", fedavg_model), ("FedProx", fedprox_model)]:
    acc, prec, rec, f1, y_true, y_pred = evaluate(model, test_loader)
    results[name] = dict(accuracy=acc, precision=prec, recall=rec, f1=f1,
                         y_true=y_true, y_pred=y_pred)
    print(f"\n{name}:")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


# ── 13. Plots ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("FedAvg vs FedProx — Potato Disease Classification", fontsize=15)

metrics = ["accuracy", "precision", "recall", "f1"]
labels  = ["Accuracy", "Precision", "Recall", "F1-Score"]

for ax, metric, label in zip(axes.flatten(), metrics, labels):
    ax.plot(fedavg_hist["round"],  fedavg_hist[metric],
            marker='o', label="FedAvg", linewidth=2)
    ax.plot(fedprox_hist["round"], fedprox_hist[metric],
            marker='s', label="FedProx", linewidth=2, linestyle='--')
    ax.set_title(label)
    ax.set_xlabel("Communication Round")
    ax.set_ylabel(label)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(left=1)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_curves.png")


# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(res["y_true"], res["y_pred"])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"{name} — Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrices.png")


# Summary bar chart
summary_df = pd.DataFrame({
    "Algorithm": ["FedAvg", "FedProx"],
    "Accuracy" : [results["FedAvg"]["accuracy"],  results["FedProx"]["accuracy"]],
    "Precision": [results["FedAvg"]["precision"], results["FedProx"]["precision"]],
    "Recall"   : [results["FedAvg"]["recall"],    results["FedProx"]["recall"]],
    "F1-Score" : [results["FedAvg"]["f1"],        results["FedProx"]["f1"]],
})
print("\nSummary Table:")
print(summary_df.to_string(index=False))

summary_df.set_index("Algorithm")[["Accuracy","Precision","Recall","F1-Score"]]\
          .plot(kind='bar', figsize=(9, 5), ylim=(0, 1.1), rot=0,
                color=['#4C72B0','#DD8452','#55A868','#C44E52'])
plt.title("Final Performance Comparison — FedAvg vs FedProx")
plt.ylabel("Score")
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig("summary_bar.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: summary_bar.png")


# ── 14. Save models ──────────────────────────────────────────
torch.save(fedavg_model.state_dict(),  "fedavg_potato.pth")
torch.save(fedprox_model.state_dict(), "fedprox_potato.pth")
print("\nModels saved: fedavg_potato.pth | fedprox_potato.pth")
print("\nDone ✓")

In [ ]:
# ============================================================
#  Federated Learning — FedMA
#  Dataset   : Potato Plant Diseases Data (Kaggle)
#  Paper ref : "Federated Learning with Matched Averaging"
#              Wang et al., ICLR 2020
#  Note      : Run AFTER federated_learning_potato.py OR
#              paste everything into one notebook sequentially.
#              This file is self-contained — it re-uses the
#              same Config, data-loading, and helpers defined
#              below so it can also run standalone.
# ============================================================

# ── 0. Imports ───────────────────────────────────────────────
import os, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from scipy.optimize import linear_sum_assignment   # Hungarian algorithm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset
import torchvision.transforms as transforms
from torchvision import datasets, models

from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, classification_report)

warnings.filterwarnings("ignore")
torch.manual_seed(42);  np.random.seed(42);  random.seed(42)


# ── 1. Config ────────────────────────────────────────────────
class Config:
    DATA_DIR        = "/kaggle/input/datasets/hafiznouman786/potato-plant-diseases-data"
    NUM_CLIENTS     = 5
    DIRICHLET_ALPHA = 0.5
    NUM_ROUNDS      = 10
    LOCAL_EPOCHS    = 5
    BATCH_SIZE      = 32
    LR              = 0.001
    MOMENTUM        = 0.9
    DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    IMG_SIZE        = 224

cfg = Config()
print(f"Device : {cfg.DEVICE}")


# ── 2. Find dataset root ─────────────────────────────────────
def find_image_root(base):
    for root, dirs, _ in os.walk(base):
        subdirs = [d for d in dirs if not d.startswith('.')]
        if subdirs:
            sample = os.path.join(root, subdirs[0])
            imgs = [f for f in os.listdir(sample)
                    if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]
            if imgs:
                return root
    return base

DATA_ROOT = find_image_root(cfg.DATA_DIR)
print(f"Image root : {DATA_ROOT}")


# ── 3. Transforms ────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
test_tf = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])


# ── 4. Dataset ───────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from PIL import Image

full_ds    = datasets.ImageFolder(root=DATA_ROOT, transform=train_tf)
CLASS_NAMES = full_ds.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Total samples: {len(full_ds)}")

targets  = np.array(full_ds.targets)
all_idx  = np.arange(len(targets))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42, stratify=targets)

class SubsetWithTransform(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        path, label = self.dataset.samples[self.indices[idx]]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

test_dataset  = SubsetWithTransform(full_ds, test_idx,  transform=test_tf)
train_dataset = SubsetWithTransform(full_ds, train_idx, transform=train_tf)
print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)}")


# ── 5. Dirichlet partition ───────────────────────────────────
def dirichlet_partition(dataset, num_clients, alpha, num_classes):
    targets = np.array([full_ds.targets[dataset.indices[i]]
                        for i in range(len(dataset))])
    class_idx = defaultdict(list)
    for i, t in enumerate(targets):
        class_idx[t].append(i)

    client_idx = defaultdict(list)
    for cls in range(num_classes):
        idxs = np.array(class_idx[cls]);  np.random.shuffle(idxs)
        props = np.random.dirichlet(np.repeat(alpha, num_clients))
        props = np.maximum(props, 1e-6);  props /= props.sum()
        splits = (props * len(idxs)).astype(int)
        splits[-1] = len(idxs) - splits[:-1].sum()
        splits = np.maximum(splits, 0)
        start = 0
        for cid, cnt in enumerate(splits):
            client_idx[cid].extend(idxs[start:start+cnt].tolist())
            start += cnt
    return dict(client_idx)

client_map = dirichlet_partition(train_dataset, cfg.NUM_CLIENTS,
                                 cfg.DIRICHLET_ALPHA, NUM_CLASSES)

print("\n── Client distribution ──")
for cid, idxs in client_map.items():
    lbls = [full_ds.targets[train_dataset.indices[i]] for i in idxs]
    dist = {CLASS_NAMES[c]: lbls.count(c) for c in range(NUM_CLASSES)}
    print(f"  Client {cid}: {len(idxs)} samples | {dist}")

def get_loader(cid):
    return DataLoader(Subset(train_dataset, client_map[cid]),
                      batch_size=cfg.BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True)

client_loaders = {cid: get_loader(cid) for cid in range(cfg.NUM_CLIENTS)}
test_loader    = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)


# ── 6. Model ─────────────────────────────────────────────────
def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m.to(cfg.DEVICE)


# ── 7. Evaluation ────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbs in loader:
            imgs = imgs.to(cfg.DEVICE)
            out  = model(imgs)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(lbs.numpy())
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average='weighted', zero_division=0)
    rec  = recall_score(labels, preds, average='weighted', zero_division=0)
    f1   = f1_score(labels, preds, average='weighted', zero_division=0)
    return acc, prec, rec, f1, labels, preds


# ── 8. Local training (standard SGD) ─────────────────────────
def local_train(model, loader, epochs):
    model.train()
    opt = optim.SGD(model.parameters(), lr=cfg.LR, momentum=cfg.MOMENTUM)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(cfg.DEVICE), labels.to(cfg.DEVICE)
            opt.zero_grad()
            crit(model(imgs), labels).backward()
            opt.step()
    return model.state_dict()


# ══════════════════════════════════════════════════════════════
#  FedMA Core — Layer-wise Neuron Matching via Hungarian Algo
# ══════════════════════════════════════════════════════════════

def _cosine_cost(A: torch.Tensor, B: torch.Tensor) -> np.ndarray:
    """
    Cost matrix (1 - cosine_similarity) between rows of A and rows of B.
    A : [n, d]   B : [m, d]
    Returns ndarray [n, m]
    """
    A = A.float();  B = B.float()
    A_n = A / (A.norm(dim=1, keepdim=True) + 1e-8)
    B_n = B / (B.norm(dim=1, keepdim=True) + 1e-8)
    sim = A_n @ B_n.T                          # [n, m]
    return (1.0 - sim).cpu().numpy()


def _match_and_merge_weights(
    global_weights: torch.Tensor,
    client_weights_list: list,
    client_sizes: list
) -> tuple:
    """
    Match neurons of each client layer to the global reference using the
    Hungarian algorithm, then compute a weighted average in matched order.

    global_weights      : [n_neurons, ...]  reference (global model)
    client_weights_list : list of tensors, each [n_neurons, ...]
    client_sizes        : sample counts per client (for weighted avg)

    Returns
    -------
    merged  : [n_neurons, ...]  averaged tensor after matching
    perms   : list of permutation arrays (one per client, shape [n_neurons])
    """
    n = global_weights.shape[0]
    G = global_weights.reshape(n, -1)          # [n, d]
    total = sum(client_sizes)
    merged = torch.zeros_like(G, dtype=torch.float32)
    perms  = []

    for sd_w, sz in zip(client_weights_list, client_sizes):
        C = sd_w.reshape(n, -1)                # [n, d]
        cost = _cosine_cost(G, C)              # [n, n]
        row_idx, col_idx = linear_sum_assignment(cost)
        perm = col_idx                         # permutation: global[i] ← client[perm[i]]
        perms.append(perm)
        # Rearrange client neurons to match global ordering
        C_matched = C[perm]                    # [n, d]
        merged += C_matched.float() * (sz / total)

    merged = merged.reshape(global_weights.shape)
    return merged, perms


def _apply_perm_to_bias(bias: torch.Tensor,
                         perm: np.ndarray) -> torch.Tensor:
    """Permute bias vector using the matched neuron order."""
    return bias[perm]


def fedma_aggregate(global_sd: dict,
                    client_sds: list,
                    client_sizes: list) -> dict:
    """
    FedMA aggregation for a ResNet18:
      • For every Conv / BN / Linear layer (except the final classifier):
          1. Match neurons from each client to the global reference
             via Hungarian algorithm on cosine-distance cost matrix.
          2. Weighted-average the matched weights.
      • Final FC layer: standard weighted average (no permutation needed
        since output neurons = fixed class indices).

    Note on ResNet shortcut / residual connections
    -----------------------------------------------
    ResNet skip connections couple layer permutations across branches.
    Full FedMA handles this with joint optimisation; here we adopt the
    practical approximation used in most FL papers: match each layer
    independently and propagate the same permutation to the paired BN
    and the NEXT layer's input dimension.  This gives the key benefit
    of neuron matching without requiring a custom ResNet solver.
    """
    new_sd   = copy.deepcopy(global_sd)
    keys     = list(global_sd.keys())

    # Group keys by layer prefix (everything before '.weight' / '.bias')
    # We process weight tensors; bias / BN stats follow their paired weight.
    processed_perms: dict = {}     # layer_prefix -> list of perms (one per client)

    i = 0
    while i < len(keys):
        key = keys[i]

        # ── Final FC layer: plain weighted average ───────────────
        if key == 'fc.weight':
            total = sum(client_sizes)
            fc_w  = torch.zeros_like(global_sd['fc.weight'], dtype=torch.float32)
            fc_b  = torch.zeros_like(global_sd['fc.bias'],   dtype=torch.float32)
            for sd, sz in zip(client_sds, client_sizes):
                fc_w += sd['fc.weight'].float() * (sz / total)
                fc_b += sd['fc.bias'].float()   * (sz / total)
            new_sd['fc.weight'] = fc_w
            new_sd['fc.bias']   = fc_b
            i += 2
            continue

        # ── Conv weight ──────────────────────────────────────────
        if key.endswith('.weight') and 'bn' not in key and 'downsample.1' not in key:
            w_key  = key
            prefix = key[:-len('.weight')]
            b_key  = prefix + '.bias'

            gw = global_sd[w_key]                     # [out, in, kH, kW] or [out, in]
            out_ch = gw.shape[0]

            # Only match if layer is large enough (skip 1-neuron layers)
            if out_ch < 2:
                i += 1
                continue

            client_ws = [sd[w_key] for sd in client_sds]
            merged_w, perms = _match_and_merge_weights(gw, client_ws, client_sizes)
            new_sd[w_key] = merged_w
            processed_perms[prefix] = perms

            # Paired conv bias (rare in ResNet but handle if present)
            if b_key in global_sd:
                total = sum(client_sizes)
                mb = torch.zeros_like(global_sd[b_key], dtype=torch.float32)
                for sd, sz, perm in zip(client_sds, client_sizes, perms):
                    mb += _apply_perm_to_bias(sd[b_key], perm).float() * (sz / total)
                new_sd[b_key] = mb
                if b_key in keys:
                    i += 1   # skip bias key in outer loop

            i += 1
            continue

        # ── BatchNorm weight / bias / running stats ───────────────
        # Find the conv prefix this BN is paired with
        if 'bn' in key or 'downsample.1' in key:
            # Derive paired conv prefix
            # e.g. layer1.0.bn1  →  layer1.0.conv1
            bn_prefix = key.rsplit('.', 1)[0]          # e.g. 'layer1.0.bn1'
            conv_prefix = bn_prefix.replace('bn', 'conv').replace('downsample.1','downsample.0')

            if conv_prefix in processed_perms:
                perms  = processed_perms[conv_prefix]
                total  = sum(client_sizes)

                for stat in ['weight','bias','running_mean','running_var']:
                    skey = bn_prefix + '.' + stat
                    if skey not in global_sd:
                        continue
                    merged_stat = torch.zeros_like(global_sd[skey], dtype=torch.float32)
                    for sd, sz, perm in zip(client_sds, client_sizes, perms):
                        if skey in sd:
                            merged_stat += sd[skey][perm].float() * (sz / total)
                    new_sd[skey] = merged_stat

                # num_batches_tracked: take max
                nb_key = bn_prefix + '.num_batches_tracked'
                if nb_key in global_sd:
                    nb_vals = [sd[nb_key] for sd in client_sds if nb_key in sd]
                    if nb_vals:
                        new_sd[nb_key] = max(nb_vals, key=lambda x: x.item())

            else:
                # No matched conv found — fall back to weighted average
                total = sum(client_sizes)
                if key.endswith(('.weight','.bias','.running_mean','.running_var')):
                    merged = torch.zeros_like(global_sd[key], dtype=torch.float32)
                    for sd, sz in zip(client_sds, client_sizes):
                        if key in sd:
                            merged += sd[key].float() * (sz / total)
                    new_sd[key] = merged

            i += 1
            continue

        # ── Anything else (num_batches_tracked already handled, etc.) ─
        i += 1

    return new_sd


# ── 9. FedMA training loop ───────────────────────────────────
def run_fedma():
    print("\n" + "="*60)
    print("  Running FedMA  (Hungarian neuron matching)")
    print("="*60)

    global_model = build_model()
    history = {"round":[], "accuracy":[], "precision":[], "recall":[], "f1":[]}

    for rnd in range(1, cfg.NUM_ROUNDS + 1):
        client_sds, client_sizes = [], []

        for cid in range(cfg.NUM_CLIENTS):
            local_model = build_model()
            local_model.load_state_dict(copy.deepcopy(global_model.state_dict()))
            sd = local_train(local_model, client_loaders[cid], cfg.LOCAL_EPOCHS)
            client_sds.append(sd)
            client_sizes.append(len(client_map[cid]))

        agg_sd = fedma_aggregate(global_model.state_dict(), client_sds, client_sizes)
        global_model.load_state_dict(agg_sd)

        if rnd % 5 == 0 or rnd == 1:
            acc, prec, rec, f1, _, _ = evaluate(global_model, test_loader)
            history["round"].append(rnd)
            history["accuracy"].append(acc)
            history["precision"].append(prec)
            history["recall"].append(rec)
            history["f1"].append(f1)
            print(f"  Round {rnd:3d} | Acc={acc:.4f}  Prec={prec:.4f}  "
                  f"Rec={rec:.4f}  F1={f1:.4f}")

    return global_model, history


fedma_model, fedma_hist = run_fedma()


# ── 10. Final results ────────────────────────────────────────
print("\n" + "="*60)
print("  FEDMA — FINAL TEST RESULTS")
print("="*60)
acc, prec, rec, f1, y_true, y_pred = evaluate(fedma_model, test_loader)
print(f"  Accuracy : {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall   : {rec:.4f}")
print(f"  F1-Score : {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


# ── 11. Plots ─────────────────────────────────────────────────
# Training curves
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("FedMA — Training Curves (Potato Disease)", fontsize=14)
for ax, metric, label in zip(axes.flatten(),
                              ["accuracy","precision","recall","f1"],
                              ["Accuracy","Precision","Recall","F1-Score"]):
    ax.plot(fedma_hist["round"], fedma_hist[metric],
            marker='D', color='#2ca02c', linewidth=2)
    ax.set_title(label);  ax.set_xlabel("Round");  ax.set_ylabel(label)
    ax.set_xlim(left=1);  ax.set_ylim(0, 1.05);   ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fedma_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fedma_training_curves.png")

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title("FedMA — Confusion Matrix")
ax.set_xlabel("Predicted");  ax.set_ylabel("True")
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig("fedma_confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fedma_confusion_matrix.png")

# Save model
torch.save(fedma_model.state_dict(), "fedma_potato.pth")
print("Model saved: fedma_potato.pth")
print("\nFedMA Done ✓")